Import necessary libraries (usually at the top)

In [58]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import matplotlib.ticker as mticker
import os


print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 3.0.5
NumPy: 2.5.1


Establishing a connection with sqlite3: means openning a connection to the sqlite database (db) file, and creates a db container for csv(s).


In [59]:
conn = sqlite3.connect("crimedatacsv_new.db")
print("Connection opened:", conn)

Connection opened: <sqlite3.Connection object at 0x00000283EDA03880>


Due to memory constraints, because of the large csv(s); I got a sample of 1500 rows from each of the orginal (large) datasets.

crime_sample = crime_df.sample(1500, random_state=42)

crime_sample     #Creates a new DataFrame called crime_sample.
= crime_df       #This is your original full crime dataset.
.sample(1500,    #This is a pandas function that randomly selects rows.
random_state=42) #This makes the random selection repeatable. Otherwise the same random sample would
                  not be repeatable; you would get a different random sample each time.


   crime_sample          #csv variable df_name
   .to_sql               #a pandas function that creates the table(s) in the db.
(  
    "crime_sample",      #name of table inside database
    conn,                #connection to sql dababase
    index=False,         #gets rid of the indexing
    if_exists="replace"  #if another table exist in the db with the same name replace it with this
                            one.
)


Load Dataset (CSV) into a dataframe:  df = pd.read_csv("your_file.csv") - The csv is being 'read' into a pandas DataFrame (df).


In [60]:
la_crime = pd.read_csv("la_crime_1500.csv")
local_crime = pd.read_csv("local_crime_1500.csv")
cps_income = pd.read_csv("cps_income_1500.csv")

Writes the df(s) into a table(s):  csv_variable_name.to_sql("table name", conn, index=False, if_exists="replace")- — 'it takes the DataFrame data and creates a SQLite table containing that data.'

In [61]:
la_crime.to_sql(        #csv variable df_name
    "la_crime",         #name of table inside database
    conn,               #connection to sql dababase
    index=False,        #gets rid of the indexing
    if_exists="replace" #if another table exist in the db with the same name replace it with this one.
)

local_crime.to_sql(
    "local_crime",
    conn,
    index=False,
    if_exists="replace"
)

cps_income.to_sql(
    "cps_income",
    conn,
    index=False,
    if_exists="replace"
)

# conn.close()  #'sqlite3.connect() → starts the phone call to_sql() / SQL queries → you communicate through the call conn.close() → hangs up the phone'

print("Database created successfully!") #prints if ran properly

Database created successfully!


To see the results of the connection, this query shows all the tables that exist in the db. Using pandas it returns a table(s) df name.

In [62]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", conn)

,name
0,ubi_subset
1,la_crime
2,local_crime
3,cps_income


Cleaning ('Wrangling Data'):  removing duplicates, overly excessive columns with nulls, datetime inconsistencies
and replacing table codes with title schema...

In [63]:
# # la_crime.isnull().sum()
# # la_crime = la_crime.fillna(0)
# # la_crime.info()
# la_crime.shape
la_crime.columns

Index(['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME',
       'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes',
       'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc',
       'Weapon Used Cd', 'Weapon Desc', 'Status', 'Status Desc', 'Crm Cd 1',
       'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'LOCATION', 'Cross Street', 'LAT',
       'LON'],
      dtype='str')

In [64]:
# local_crime.isnull().sum()
# local_crime = local_crime.fillna(0)
# local_crime.info()
local_crime.columns
local_crime.shape

(1500, 16)

In [65]:
# cps_income.isnull().sum()
# cps_income.info()
cps_income.columns
cps_income.shape

(1500, 16)

#Using .map to change the codes for the sexes to words: 1 - for males, and 2 - females, making the chart more readable. 

In [66]:
sex_map = {
    1: 'Male',
    2: 'Female'
}

cps_income['SEX'] = cps_income['SEX'].map(sex_map)
cps_income.head()


,YEAR,REGION,STATEFIP,COUNTY,METRO,ASECWT,AGE,SEX,RACE,OCC,UHRSWORK1,EDUC,SCHLCOLL,INCWAGE,OFFTOTVAL,POVERTY
0,2024,33,48,0,2,1898.77,80,Male,100,0,999,40,0,0,91488,23
1,2024,31,13,0,3,3187.62,64,Female,100,800,40,111,0,161000,294060,23
2,2024,12,36,0,4,2597.91,17,Female,100,0,999,60,1,0,55002,23
3,2024,21,17,17119,3,1810.53,25,Female,100,0,999,111,5,30000,43201,21
4,2024,31,37,37001,4,11338.34,45,Male,100,6600,45,71,5,55000,65000,23


Next, we remap race/ethnicity for more reabability.

In [ ]:
race_map = {
    100: "White",
    200: "Black",
    300: "American Indian/Alaska Native",
    651: "Asian",
    652: "Native Hawaiian/Pacific Islander",

    801: "White & Black",
    802: "White & American Indian",
    803: "Black & American Indian",
    804: "White & Asian",
    805: "Black & Asian",
    806: "American Indian & Asian",
    807: "White, Black & American Indian",
    808: "White, Black & Asian",
    809: "White, American Indian & Asian",
    810: "Black, American Indian & Asian",
    811: "White & Pacific Islander",
    812: "Black & Pacific Islander",
    813: "Asian & Pacific Islander",
    814: "American Indian & Pacific Islander",
    815: "White, Black & Pacific Islander",
    816: "White, Asian & Pacific Islander",
    817: "Black, Asian & Pacific Islander",
    818: "White, American Indian & Pacific Islander",
    819: "Black, American Indian & Pacific Islander",
    820: "American Indian, Asian & Pacific Islander",
    830: "Other Multiple Race"
}

cps_income['RACE'] = cps_income['RACE'].map(race_map)
cps_income.head(50)


,YEAR,REGION,STATEFIP,COUNTY,METRO,ASECWT,AGE,SEX,RACE,OCC,UHRSWORK1,EDUC,SCHLCOLL,INCWAGE,OFFTOTVAL,POVERTY
0,2024,33,48,0,2,1898.77,80,Male,White,0,999,40,0,0,91488,23
1,2024,31,13,0,3,3187.62,64,Female,White,800,40,111,0,161000,294060,23
2,2024,12,36,0,4,2597.91,17,Female,White,0,999,60,1,0,55002,23
3,2024,21,17,17119,3,1810.53,25,Female,White,0,999,111,5,30000,43201,21
4,2024,31,37,37001,4,11338.34,45,Male,White,6600,45,71,5,55000,65000,23
5,2024,21,18,0,2,4448.76,54,Female,White,4230,999,123,5,12000,133000,23
6,2024,31,11,11001,2,419.87,41,Male,White,410,28,81,5,45429,45460,23
7,2024,33,40,0,3,3311.47,40,Male,White,4710,16,111,5,45000,143953,23
8,2024,42,6,6007,4,2336.91,77,Female,Asian,0,999,123,0,0,205186,23
9,2024,32,47,0,1,1177.87,13,Male,White,0,999,1,0,99999999,126963,23


Here, Occupational Schema...using Python and and its functions...

In [ ]:
occ_map = {} #creates an empty Python dictionary

with open("occ_codebook.txt", "r") as file: #opening the text file, and reads it
    lines = [line.strip() for line in file.readlines() if line.strip()] #list comprehension-every line being read,
                                                                         #removes extra spaces and newline characters.

for i in range(0, len(lines), 2):  #loop through list
    code = int(lines[i]) #convert occupation code into integer
    title = lines[i + 1] #gets the occupation name
    occ_map[code] = title  #creates a dictionary entry

cps_income['OCC'] = cps_income['OCC'].map(occ_map)
cps_income.head(50)



,YEAR,REGION,STATEFIP,COUNTY,METRO,ASECWT,AGE,SEX,RACE,OCC,UHRSWORK1,EDUC,SCHLCOLL,INCWAGE,OFFTOTVAL,POVERTY
0,2024,33,48,0,2,1898.77,80,Male,White,Not in universe,999,40,0,0,91488,23
1,2024,31,13,0,3,3187.62,64,Female,White,Accountants and auditors,40,111,0,161000,294060,23
2,2024,12,36,0,4,2597.91,17,Female,White,Not in universe,999,60,1,0,55002,23
3,2024,21,17,17119,3,1810.53,25,Female,White,Not in universe,999,111,5,30000,43201,21
4,2024,31,37,37001,4,11338.34,45,Male,White,"Helpers, construction trades",45,71,5,55000,65000,23
5,2024,21,18,0,2,4448.76,54,Female,White,Maids and housekeeping cleaners,999,123,5,12000,133000,23
6,2024,31,11,11001,2,419.87,41,Male,White,"Property, real estate, and community associati...",28,81,5,45429,45460,23
7,2024,33,40,0,3,3311.47,40,Male,White,First-line supervisors/managers of non-retail ...,16,111,5,45000,143953,23
8,2024,42,6,6007,4,2336.91,77,Female,Asian,Not in universe,999,123,0,0,205186,23
9,2024,32,47,0,1,1177.87,13,Male,White,Not in universe,999,1,0,99999999,126963,23


#School/College....

In [ ]:
schlcoll_map = {}

with open("schlcoll_codebook.txt", "r", encoding="utf-8") as file:
    lines = [line.strip() for line in file if line.strip()]

for i in range(0, len(lines), 2):
    code = int(lines[i])
    description = lines[i + 1]
    schlcoll_map[code] = description

# Map the values
cps_income["SCHLCOLL"] = cps_income["SCHLCOLL"].map(schlcoll_map)

cps_income.head()






,YEAR,REGION,STATEFIP,COUNTY,METRO,ASECWT,AGE,SEX,RACE,OCC,UHRSWORK1,EDUC,SCHLCOLL,INCWAGE,OFFTOTVAL,POVERTY
0,2024,33,48,0,2,1898.77,80,Male,White,Not in universe,999,40,NaN,0,91488,23
1,2024,31,13,0,3,3187.62,64,Female,White,Accountants and auditors,40,111,NaN,161000,294060,23
2,2024,12,36,0,4,2597.91,17,Female,White,Not in universe,999,60,Not enrolled,0,55002,23
3,2024,21,17,17119,3,1810.53,25,Female,White,Not in universe,999,111,College/university full time,30000,43201,21
4,2024,31,37,37001,4,11338.34,45,Male,White,"Helpers, construction trades",45,71,College/university full time,55000,65000,23


In [ ]:
#Normaliztion of educ_codebook.txt file...so it can be used to 'map' educ codebook...

In [ ]:
#AI suggested format

educ_input = "educ_codebook.txt"
educ_output = "educ_codebook_normalized.txt"

with open(educ_input, "r", encoding="utf-8") as infile:
    lines = [line.strip() for line in infile if line.strip()]

# Remove header
if not lines[0].isdigit():
    lines = lines[1:]

normalized = []

for i in range(0, len(lines), 2):
    code = lines[i]
    description = lines[i + 1]

    # Check if code is a range (example: 010–014)
    if "–" in code:
        start, end = code.split("–")

        for num in range(int(start), int(end) + 1):
            normalized.append(f"{num:03d}")
            normalized.append(description)

    # Single code
    else:
        normalized.append(code)
        normalized.append(description)


# Write the new normalized file
with open(educ_output, "w", encoding="utf-8") as outfile:
    for line in normalized:
        outfile.write(line + "\n")

print("Normalized education codebook created!")


Normalized education codebook created!


In [ ]:
educ_map = {}

with open('educ_codebook_normalized.txt', "r", encoding="utf-8") as file:
    lines = [line.strip() for line in file if line.strip()]

for i in range(0, len(lines), 2):
    code = int(lines[i])
    description = lines[i + 1]
    educ_map[code] = description

# Map the values
cps_income["EDUC"] = cps_income["EDUC"].astype(int)
cps_income['EDUC'] = cps_income['EDUC'].map(educ_map)

cps_income["INCWAGE"] = cps_income["INCWAGE"].replace(99999999, np.nan)

cps_income.head(100)


,YEAR,REGION,STATEFIP,COUNTY,METRO,ASECWT,AGE,SEX,RACE,OCC,UHRSWORK1,EDUC,SCHLCOLL,INCWAGE,OFFTOTVAL,POVERTY
0,2024,33,48,0,2,1898.77,80,Male,White,Not in universe,999,Grades 9–12,NaN,0.0,91488,23
1,2024,31,13,0,3,3187.62,64,Female,White,Accountants and auditors,40,NaN,NaN,161000.0,294060,23
2,2024,12,36,0,4,2597.91,17,Female,White,Not in universe,999,Grades 16–18 (graduate),Not enrolled,0.0,55002,23
3,2024,21,17,17119,3,1810.53,25,Female,White,Not in universe,999,NaN,College/university full time,30000.0,43201,21
4,2024,31,37,37001,4,11338.34,45,Male,White,"Helpers, construction trades",45,Grades 19–21 (postgraduate),College/university full time,55000.0,65000,23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2024,31,13,0,1,2176.55,53,Female,Black,Physician assistants,999,NaN,College/university full time,0.0,79000,23
96,2024,41,32,0,2,2249.17,59,Male,White,Secondary school teachers,45,NaN,NaN,105000.0,314985,23
97,2024,41,16,0,4,864.72,15,Female,White,Child care workers,15,Grades 16–18 (graduate),NaN,10000.0,186858,23
98,2024,41,4,4013,2,5394.68,7,Female,White,Not in universe,999,"None, preschool, or kindergarten",NaN,NaN,0,10


Saved normalized/cleaned file...

In [ ]:
cps_income.to_parquet("cps_income_clean.parquet", index=False)

# df = pd.read_parquet("cps_income_clean.parquet")

#Creating ubi subset columns...

In [ ]:
cps_income[['EDUC','SCHLCOLL','INCWAGE','AGE','SEX','RACE','OCC']].head()
UBI_subset = cps_income[['EDUC','SCHLCOLL','INCWAGE','AGE','SEX','RACE','OCC']]
UBI_subset.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   EDUC      749 non-null    str    
 1   SCHLCOLL  728 non-null    str    
 2   INCWAGE   1218 non-null   float64
 3   AGE       1500 non-null   int64  
 4   SEX       1500 non-null   str    
 5   RACE      1500 non-null   str    
 6   OCC       1294 non-null   str    
dtypes: float64(1), int64(1), str(5)
memory usage: 167.3 KB


In [ ]:
#Cleaning ubi subset 'INCWAGE' column: removing place holders, with '0'
#Dropping nulls

UBI_subset['INCWAGE'] = UBI_subset['INCWAGE'].replace(99999999, np.nan)
UBI_subset = UBI_subset.dropna(subset=['INCWAGE'])
UBI_subset = UBI_subset[UBI_subset['INCWAGE'] > 0]
UBI_subset.head(100)

,EDUC,SCHLCOLL,INCWAGE,AGE,SEX,RACE,OCC
1,NaN,NaN,161000.0,64,Female,White,Accountants and auditors
3,NaN,College/university full time,30000.0,25,Female,White,Not in universe
4,Grades 19–21 (postgraduate),College/university full time,55000.0,45,Male,White,"Helpers, construction trades"
5,NaN,College/university full time,12000.0,54,Female,White,Maids and housekeeping cleaners
6,Grades 22–24 (doctoral),College/university full time,45429.0,41,Male,White,"Property, real estate, and community associati..."
...,...,...,...,...,...,...,...
204,NaN,College/university full time,87000.0,38,Male,White,Compliance officers
205,NaN,College/university full time,100000.0,32,Male,White,"Industrial engineers, including health and safety"
209,NaN,NaN,98000.0,61,Male,White,NaN
210,NaN,NaN,3000.0,64,Male,White,Not in universe


In [ ]:
#Creating a subset table to query...

In [ ]:
UBI_subset = UBI_subset[['EDUC','SCHLCOLL','INCWAGE','AGE','SEX','RACE','OCC']]
UBI_subset.to_sql("ubi_subset", conn, if_exists="replace", index=False)



718

#SQL Queries...

In [ ]:
df = pd.read_sql("""
SELECT *
FROM la_crime;
""", conn)

df

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON
0,230514082,09/28/2023 12:00:00 AM,08/18/2023 12:00:00 AM,1800,5,Harbor,514,1,310,BURGLARY,...,IC,Invest Cont,310.0,NaN,NaN,None,1200 W PACIFIC COAST HY,NaN,33.7904,-118.2777
1,221311915,05/27/2022 12:00:00 AM,05/27/2022 12:00:00 AM,145,13,Newton,1373,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,IC,Invest Cont,230.0,998.0,NaN,None,56TH ST,CENTRAL ST,33.9916,-118.2564
2,231017635,12/27/2023 12:00:00 AM,12/26/2023 12:00:00 AM,800,10,West Valley,1067,2,740,"VANDALISM - FELONY ($400 & OVER, ALL CHURCH VA...",...,IC,Invest Cont,740.0,NaN,NaN,None,17400 VENTURA BL,NaN,34.1660,-118.5095
3,200211304,06/13/2020 12:00:00 AM,06/13/2020 12:00:00 AM,2300,2,Rampart,295,1,236,INTIMATE PARTNER - AGGRAVATED ASSAULT,...,AA,Adult Arrest,236.0,NaN,NaN,None,1300 CONSTANCE ST,NaN,34.0451,-118.2779
4,241206655,02/14/2024 12:00:00 AM,02/14/2024 12:00:00 AM,1600,12,77th Street,1208,2,930,CRIMINAL THREATS - NO WEAPON DISPLAYED,...,IC,Invest Cont,930.0,NaN,NaN,None,600 W VERNON AV,NaN,34.0038,-118.2842
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,201111704,07/22/2020 12:00:00 AM,07/22/2020 12:00:00 AM,50,11,Northeast,1159,1,310,BURGLARY,...,AA,Adult Arrest,310.0,998.0,NaN,None,4500 N FIGUEROA ST,NaN,34.1009,-118.2036
1496,241405191,01/28/2024 12:00:00 AM,01/28/2024 12:00:00 AM,1210,14,Pacific,1443,2,745,VANDALISM - MISDEAMEANOR ($399 OR UNDER),...,IC,Invest Cont,745.0,NaN,NaN,None,VENICE BL,GRAND VIEW,33.9928,-118.4513
1497,211907685,04/15/2021 12:00:00 AM,04/15/2021 12:00:00 AM,1430,19,Mission,1974,2,624,BATTERY - SIMPLE ASSAULT,...,IC,Invest Cont,624.0,NaN,NaN,None,8900 KESTER AV,NaN,34.2318,-118.4575
1498,241807596,03/16/2024 12:00:00 AM,03/16/2024 12:00:00 AM,2055,18,Southeast,1838,1,761,BRANDISH WEAPON,...,IC,Invest Cont,761.0,NaN,NaN,None,10300 WILMINGTON AV,NaN,33.9432,-118.2391


In [ ]:
pd.read_sql("""
SELECT "DATE OCC",
       "TIME OCC",
       LOCATION    
FROM la_crime;
""", conn)

,DATE OCC,TIME OCC,LOCATION
0,08/18/2023 12:00:00 AM,1800,1200 W PACIFIC COAST HY
1,05/27/2022 12:00:00 AM,145,56TH ST
2,12/26/2023 12:00:00 AM,800,17400 VENTURA BL
3,06/13/2020 12:00:00 AM,2300,1300 CONSTANCE ST
4,02/14/2024 12:00:00 AM,1600,600 W VERNON AV
...,...,...,...
1495,07/22/2020 12:00:00 AM,50,4500 N FIGUEROA ST
1496,01/28/2024 12:00:00 AM,1210,VENICE BL
1497,04/15/2021 12:00:00 AM,1430,8900 KESTER AV
1498,03/16/2024 12:00:00 AM,2055,10300 WILMINGTON AV


In [ ]:
pd.read_sql("""
SELECT
     LOCATION,
    "Weapon Desc", COUNT(*) AS occurrences
FROM 
    la_crime
WHERE
     "Weapon Desc" IS NOT NULL
GROUP BY 
    LOCATION, "Weapon Desc"
ORDER BY
     occurrences DESC
LIMIT 15;
""", conn)

,LOCATION,Weapon Desc,occurrences
0,100 S FIGUEROA ST,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",2
1,1800 W SLAUSON AV,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",2
2,4000 LAUREL CANYON BL,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",2
3,6TH ST,KNIFE WITH BLADE 6INCHES OR LESS,2
4,BROADWAY,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",2
5,CENTRAL AV,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",2
6,100 THE GROVE DR,VERBAL THREAT,1
7,100 E 11TH ST,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",1
8,100 S SERRANO AV,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",1
9,1000 BLAINE ST,SIMULATED GUN,1


Where is the MOST crime?

In [ ]:
pd.read_sql("""
SELECT
    LOCATION,
    COUNT(*) AS crime_count
FROM la_crime
GROUP BY LOCATION
ORDER BY crime_count DESC
LIMIT 1;
""", conn)

,LOCATION,crime_count
0,6TH ST,6


What time of day does the MOST crime occur?

What crimes are occuring the most?

Areas with HIGHEST crime RATES?

What areas are experiecing a specific crime:  burgalries, theft, robbery...

What areas are experiencing the lowest crime rates?

Average income:  by sex, race, age, education.....

In [ ]:
#avg icome by sex
avg_income = pd.read_sql(
    """
    SELECT
        SEX,
    ROUND(AVG(INCWAGE),2) AS avg_income
    FROM ubi_subset
    GROUP BY SEX;
    """,
    conn
)

avg_income

,SEX,avg_income
0,Female,53390.35
1,Male,84378.41


In [ ]:
#avg icome by race
avg_income = pd.read_sql(
    """
    SELECT
        RACE,
    ROUND(AVG(INCWAGE),2) AS avg_income
    FROM ubi_subset
    GROUP BY RACE
    ORDER BY AVG_INCOME DESC;
    """,
    conn
)

avg_income

,RACE,avg_income
0,Asian,97443.24
1,Black & American Indian,87331.50
2,White,70081.86
3,White & Asian,63509.50
4,Black,56972.78
5,Asian & Pacific Islander,52000.00
6,American Indian/Alaska Native,49527.27
7,White & Black,49333.33
8,White & American Indian,48048.57
9,Black & Pacific Islander,35000.00


In [ ]:
#avg icome by age
avg_income = pd.read_sql(
    """
    SELECT
        AGE,
    ROUND(AVG(INCWAGE),2) AS avg_income
    FROM ubi_subset
    GROUP BY AGE;
    """,
    conn
)

avg_income

,AGE,avg_income
0,15,15200.00
1,16,11875.00
2,17,7036.67
3,18,11500.00
4,19,27544.44
...,...,...
61,76,18000.00
62,77,13500.00
63,78,11000.00
64,80,73600.00


In [76]:
#avg icome by education
avg_income = pd.read_sql(
    """
    SELECT
        EDUC,
    ROUND(AVG(INCWAGE),2) AS avg_income
    FROM ubi_subset
    GROUP BY EDUC
    ORDER BY EDUC ASC;
    """,
    conn
)

avg_income

,EDUC,avg_income
0,NaN,76845.00
1,Grades 13–15 (postsecondary),26055.56
2,Grades 16–18 (graduate),27741.00
3,Grades 19–21 (postgraduate),35333.33
4,Grades 1–4,42066.67
5,Grades 22–24 (doctoral),53897.41
6,Grades 25–27 (professional),58951.17
7,Grades 5–6,14250.00
8,Grades 7–8,208333.17
9,Grades 9–12,25688.89


Correlation Heatmap

Geographic Crime Map

In [ ]:
# import folium

# m = folium.Map(location=[39,-95], zoom_start=11)

# for _, row in la_crime.iterrows():
#     folium.CircleMarker(
#         [row['LAT'], row['LON']],
#         radius=3,
#         color="red"
#     ).add_to(m)

# m